In [1]:
import os, sys, platform, yaml, re
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, StringType
from pathlib import Path
import matplotlib.pyplot as plt
import hmac
import hashlib

In [2]:
conf_path = str(Path.cwd() / "config" / "ETL_config.yaml")
with open(conf_path, "r") as f:
    CFG = yaml.safe_load(f)

IS_WIN          = platform.system() == "Windows"
CSV_DIR         = CFG["paths"]["csv_base_dir"]["windows" if IS_WIN else "linux"]
PG_URL          = CFG["postgres"]["url"]
PG_USER         = CFG["postgres"]["user"]
PG_PASS         = CFG["postgres"]["pass"]
PG_SCHEMA       = CFG["postgres"]["schema_out"]["schema_name"]
PG_TABLE1       = CFG["postgres"]["schema_out"]["table1"]
PG_TABLE2       = CFG["postgres"]["schema_out"]["table2"]
PG_TABLE3       = CFG["postgres"]["schema_out"]["table3"]
FILES           = CFG["csv"]["files"]
NUM_PARTITIONS  = CFG["csv"]["num_partitions"]
JDBC_BATCHSIZE  = CFG["postgres"]["batchsize"]
JDBC_FETCHSIZE  = CFG["postgres"]["fetchsize"]
SPARK_LOCAL_DIR = CFG["spark"]["local_dirs"]["windows" if IS_WIN else "linux"]
NEO4J_URI  = CFG["neo4j"]["uri"]          # "bolt://localhost:7687"
NEO4J_USER = CFG["neo4j"]["user"]
NEO4J_PASS = CFG["neo4j"]["pass"]
NEO4J_DB   = CFG["neo4j"]["database"]

opts = {
    "url": NEO4J_URI,
    "authentication.type": "basic",
    "authentication.basic.username": NEO4J_USER,
    "authentication.basic.password": NEO4J_PASS,
    "database": NEO4J_DB,
}

os.environ["PYSPARK_PYTHON"] = sys.executable       # usa el Python del kernel actual
os.environ["JAVA_HOME"] = os.environ.get("JAVA_HOME", "/usr/lib/jvm/java-17-openjdk-amd64") #Chequear versión Windows después
Path(SPARK_LOCAL_DIR).mkdir(parents=True, exist_ok=True)

In [3]:
builder = (
    SparkSession.builder
    .master("local[*]")  
    .appName(CFG["spark"]["app_name"])
    .config("spark.sql.shuffle.partitions", str(CFG["spark"]["shuffle_partitions"]))
    .config("spark.driver.memory", CFG["spark"]["driver_memory"])
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.local.dir", SPARK_LOCAL_DIR)   
    .config(
        "spark.jars.packages",
        ",".join([
            "org.postgresql:postgresql:42.7.4",
            "org.neo4j:neo4j-connector-apache-spark_2.12:5.3.10_for_spark_3"
        ])
    )
)

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

jdbc_props = {
        "user": PG_USER,
        "password": PG_PASS,
        "driver": "org.postgresql.Driver",
        "fetchsize": str(JDBC_FETCHSIZE)
    }

25/12/03 12:19:44 WARN Utils: Your hostname, AsusMare resolves to a loopback address: 127.0.1.1; using 192.168.1.12 instead (on interface wlp2s0)
25/12/03 12:19:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/opt/spark-3.5.3-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/felpipe/.ivy2/cache
The jars for the packages stored in: /home/felpipe/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.neo4j#neo4j-connector-apache-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-102ac73c-cfcb-4366-be42-0dfddd83ba24;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
	found org.neo4j#neo4j-connector-apache-spark_2.12;5.3.10_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.12_common;5.3.10_for_spark_3 in central
	found org.neo4j#caniuse-core;1.3.0 in local-m2-cache
	found org.neo4j#caniuse-api;1.3.0 in local-m2-cache
	found org.jetbrains.kotlin#kotlin-stdlib;2.1.20 in local-m2-cache
	found org.jetbrains#annotations;13.0 in local-m2-cache
	found org.neo4j#caniuse-neo4j-detection;1.3.0 in local-m2-cache
	found org.neo4j.driver#neo4j-java-driver-slim;4.4.21 in local-m2-cache
	

# Anonimización

In [4]:
df_acc = (
    spark.read.format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", f"{PG_SCHEMA}.{PG_TABLE1}")  
    .option("user", PG_USER)
    .option("password", PG_PASS)
    .option("driver", "org.postgresql.Driver")
    .option("partitionColumn", "account")
    .option("lowerBound", "1")
    .option("upperBound", "10000000")
    .option("numPartitions", "6")
    .option("fetchsize", str(JDBC_FETCHSIZE))
    .load()
)

df_txs = (
    spark.read.format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", f"{PG_SCHEMA}.{PG_TABLE2}")  
    .option("user", PG_USER)
    .option("password", PG_PASS)
    .option("driver", "org.postgresql.Driver")
    .option("partitionColumn", "id")
    .option("lowerBound", "1")
    .option("upperBound", "10000000")
    .option("numPartitions", "6")
    .option("fetchsize", str(JDBC_FETCHSIZE))
    .load()
)

In [5]:
HMAC_KEY = b"UNA_CLAVE_SECRETA_LARGA_AQUI" # 32 bytes generada con CPRING

def hmac_sha256(value: str) -> str:
    if value is None:
        return None
    elif type(value) == int:
        value = str(value)

    #Asegurar que value venga ya normalizado como string
    value_bytes = value.encode("utf-8")
    return hmac.new(HMAC_KEY, value_bytes, hashlib.sha256).hexdigest()

hmac_udf = F.udf(hmac_sha256, StringType())

In [6]:
df_hash = (
    df_acc
    .select(F.col("account").alias("num_cta"))
    .withColumn("cta_tok", hmac_udf(F.col("num_cta")))
)

(df_hash.write
 .format("jdbc")
 .option("url", PG_URL)
 .option("dbtable", f"{PG_SCHEMA}.cta_hash")
 .option("user", PG_USER)
 .option("password", PG_PASS)
 .option("driver", "org.postgresql.Driver")
 .option("batchsize", str(JDBC_BATCHSIZE))
 .option("truncate", "true") 
 .mode("overwrite")  # o 'append'
 .save())

In [8]:
df_clientes = (
    df_acc
    .select(F.col("account").alias("num_cta"), F.col("location").alias("bco_cta"))
    .withColumn("num_cta", hmac_udf(F.col("num_cta")))
)

df_entrada = (
    df_txs
    .select(
        F.col("date_time").alias("event_date"),
        F.col("amount").alias("mto_trf"),
        F.col("sender_account").alias("cta_ori"),
        F.col("receiver_account").alias("cta_dst"),
        F.col("received_currency").alias("cod_mon")
    )
    .withColumn("cta_ori", hmac_udf(F.col("cta_ori")))
    .withColumn("cta_dst", hmac_udf(F.col("cta_dst")))
    .join(
        df_clientes,
        F.col("cta_dst") == F.col("num_cta"),
        "inner"
    )
    .filter(F.col("bco_cta") == F.lit("UK"))
).select(
    F.col("event_date"),
    F.col("mto_trf"),
    F.col("cta_ori"),
    F.col("cta_dst"),
    F.col("cod_mon"),
    F.col("bco_cta").alias("bco_dst")
)

df_salida = (
    df_txs
    .select(
        F.col("date_time").alias("event_date"),
        F.col("amount").alias("mto_trf"),
        F.col("sender_account").alias("cta_ori"),
        F.col("receiver_account").alias("cta_dst"),
        F.col("received_currency").alias("cod_mon")
    )
    .withColumn("cta_ori", hmac_udf(F.col("cta_ori")))
    .withColumn("cta_dst", hmac_udf(F.col("cta_dst")))
    .join(
        df_clientes,
        F.col("cta_ori") == F.col("num_cta"),
        "inner"
    )
    .filter(F.col("bco_cta") == F.lit("UK"))
).select(
    F.col("event_date"),
    F.col("mto_trf"),
    F.col("cta_ori"),
    F.col("cta_dst"),
    F.col("cod_mon"),
    F.col("bco_cta").alias("bco_ori")
)

In [9]:
(df_clientes.write
 .format("jdbc")
 .option("url", PG_URL)
 .option("dbtable", f"{PG_SCHEMA}.cc")
 .option("user", PG_USER)
 .option("password", PG_PASS)
 .option("driver", "org.postgresql.Driver")
 .option("batchsize", str(JDBC_BATCHSIZE))
 .option("truncate", "true") 
 .mode("overwrite")  # o 'append'
 .save())

In [10]:
(df_entrada.write
 .format("jdbc")
 .option("url", PG_URL)
 .option("dbtable", f"{PG_SCHEMA}.txe")
 .option("user", PG_USER)
 .option("password", PG_PASS)
 .option("driver", "org.postgresql.Driver")
 .option("batchsize", str(JDBC_BATCHSIZE))
 .option("truncate", "true") 
 .mode("overwrite")  # o 'append'
 .save())

In [11]:
(df_salida.write
 .format("jdbc")
 .option("url", PG_URL)
 .option("dbtable", f"{PG_SCHEMA}.txs")
 .option("user", PG_USER)
 .option("password", PG_PASS)
 .option("driver", "org.postgresql.Driver")
 .option("batchsize", str(JDBC_BATCHSIZE))
 .option("truncate", "true") 
 .mode("overwrite")  # o 'append'
 .save())

# Ego-Graph

In [12]:
del df_clientes, df_entrada, df_salida #Ahora los consultamos directamente de PostGres

In [13]:
df_clientes = (
    spark.read.format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", f"{PG_SCHEMA}.cc")  
    .option("user", PG_USER)
    .option("password", PG_PASS)
    .option("driver", "org.postgresql.Driver")
    #.option("partitionColumn", "num_cta") #No se puede hacer particionado a través de columnas string
    #.option("lowerBound", "1")
    #.option("upperBound", "10000000")
    #.option("numPartitions", "6")
    .option("fetchsize", str(JDBC_FETCHSIZE))
    .load()
)

In [14]:
df_entrada = (
    spark.read.format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", f"{PG_SCHEMA}.txe")  
    .option("user", PG_USER)
    .option("password", PG_PASS)
    .option("driver", "org.postgresql.Driver")
    .option("partitionColumn", "event_date")
    .option("lowerBound", "2015-01-01 00:00:00")
    .option("upperBound", "2025-01-01 00:00:00")
    .option("numPartitions", "6")
    .option("fetchsize", str(JDBC_FETCHSIZE))
    .load()
)

In [15]:
df_salida = (
    spark.read.format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", f"{PG_SCHEMA}.txs")  
    .option("user", PG_USER)
    .option("password", PG_PASS)
    .option("driver", "org.postgresql.Driver")
    .option("partitionColumn", "event_date")
    .option("lowerBound", "2015-01-01 00:00:00")
    .option("upperBound", "2025-01-01 00:00:00")
    .option("numPartitions", "6")
    .option("fetchsize", str(JDBC_FETCHSIZE))
    .load()
)

In [58]:
#Querys cypher para ego-graphs

center_candidate = """
MATCH (a:Account)
WITH a, COUNT { (a)-[:TX]-() } AS deg
WHERE deg >= 20 AND deg <= 200
RETURN a, deg
ORDER BY rand()
LIMIT 1;
"""

ego2hops = """
MATCH (center:Account)
WHERE center.account_number = 6984045785

// ---- 1er hop: vecinos directos, aleatorios y limitados ----
MATCH (center)-[:TX]-(n1:Account)
WITH center, n1
ORDER BY rand()                         // aleatorizamos el orden de los vecinos
WITH center, collect(DISTINCT n1)[0..200] AS firstHop

// ---- 2º hop: vecinos de los vecinos, también limitado ----
UNWIND firstHop AS n1
MATCH (n1)-[:TX]-(n2:Account)
WHERE n2 <> center                      // no volver al centro
  AND NOT n2 IN firstHop                // no duplicar nodos del 1er hop
WITH center, firstHop, n2
ORDER BY rand()                         // aleatorizamos qué 2-hop quedan
WITH center, firstHop,
     collect(DISTINCT n2)[0..200] AS secondHop

// ---- Traer el subgrafo (nodos + relaciones entre ellos) ----
MATCH (n:Account)
WHERE n = center OR n IN firstHop OR n IN secondHop
MATCH (n)-[r:TX]-(m:Account)
WHERE m = center OR m IN firstHop OR m IN secondHop
RETURN center,
       collect(DISTINCT n) AS nodes,
       collect(DISTINCT r) AS relationships;

"""

ego2hops = """
// Parámetros:
// $centerId
// $maxFirstHop, $maxSecondHop, $maxThirdHop

MATCH (center:Account)
WHERE id(center) = $centerId

// 1er hop
MATCH (center)-[:TX]-(n1:Account)
WITH center, n1
ORDER BY rand()
WITH center, collect(DISTINCT n1)[0..$maxFirstHop] AS firstHop

// 2º hop
UNWIND firstHop AS n1
MATCH (n1)-[:TX]-(n2:Account)
WHERE n2 <> center
  AND NOT n2 IN firstHop
WITH center, firstHop, n2
ORDER BY rand()
WITH center, firstHop,
     collect(DISTINCT n2)[0..$maxSecondHop] AS secondHop

// 3er hop
UNWIND secondHop AS n2
MATCH (n2)-[:TX]-(n3:Account)
WHERE n3 <> center
  AND NOT n3 IN firstHop
  AND NOT n3 IN secondHop
WITH center, firstHop, secondHop, n3
ORDER BY rand()
WITH center, firstHop, secondHop,
     collect(DISTINCT n3)[0..$maxThirdHop] AS thirdHop

// Subgrafo completo de 3 hops
MATCH (n:Account)
WHERE n = center OR n IN firstHop OR n IN secondHop OR n IN thirdHop
MATCH (n)-[r:TX]-(m:Account)
WHERE m = center OR m IN firstHop OR m IN secondHop OR m IN thirdHop
RETURN center,
       collect(DISTINCT n) AS nodes,
       collect(DISTINCT r) AS relationships;

"""

In [65]:
def get_ego_graph(df, center_node_cands: DataFrame | str | list, direction, n_hops) -> dict | list:
    """
    Construye el ego-graph de una cuenta hasta n_hops usando recursividad.
    Retorna un diccionario anidado representando el grafo.
    (VERSIÓN RECURSIVA)
    cc = 67cbdee2aaad33ed20ccf8c7c1f5ba3a9c4377b4d5e05d055d090565cc4937eb
    """
    # Extrae el valor de cuenta desde el DataFrame center_node si es necesario
    if hasattr(center_node_cands, "collect") or type(center_node_cands) == DataFrame:
        rows = center_node_cands.collect()
        if len(rows) == 0:
            raise ValueError("center_node_cands DataFrame está vacío.")
        # Busca la columna que contenga 'cuenta' o similar
        colname = [c for c in center_node_cands.columns if "cta" in c][0]
        center_node = rows[0][colname]
    elif type(center_node_cands) == str:
        center_node = center_node_cands
    elif type(center_node_cands) == list:
        center_node = center_node_cands[0]
    else:
        raise ValueError(f"center_node_cands debe ser DataFrame, str o list. Se recibió {type(center_node_cands)}.")

    def _to_hashable(val):
        if isinstance(val, bytearray):
            return bytes(val)
        return val

    def _ego(current_account, hops_left, visited):
        current_account_hash = _to_hashable(current_account)
        if hops_left == 0:
            return []
        if direction == 'in':
            # Encuentra las cuentas que transfirieron a current_cuenta
            accounts_df = df.filter(F.col('cta_dst') == F.lit(current_account)) \
                           .select('cta_ori') \
                           .distinct()
            accounts = [row['cta_ori'] for row in accounts_df.collect()]
            accounts = [_to_hashable(c) for c in accounts if _to_hashable(c) not in visited]
            if hops_left == 1:
                return accounts
            return {c: _ego(c, hops_left-1, visited | {current_account_hash}) for c in accounts}
        else:
            # Encuentra las cuentas a las que current_cuenta transfirió
            accounts_df = df.filter(F.col('cta_ori') == F.lit(current_account)) \
                           .select('cta_dst') \
                           .distinct()
            accounts = [row['cta_dst'] for row in accounts_df.collect()]
            accounts = [_to_hashable(c) for c in accounts if _to_hashable(c) not in visited]
            if hops_left == 1:
                return accounts
            return {c: _ego(c, hops_left-1, visited | {current_account_hash}) for c in accounts}
    
    return _ego(center_node, n_hops, set())

In [ ]:
def ego_count(ego_graph):
    """
    Cuenta el número de cuentas en un ego-grafo representado como:
      - dict: {cuenta: subgrafo o lista}
      - list: [cuentas vecinas] en el último nivel
    """
    if isinstance(ego_graph, list):
        # último nivel: solo una lista de cuentas
        return len(ego_graph)
    
    elif isinstance(ego_graph, dict):
        # este nivel: las keys son cuentas
        count = len(ego_graph)
        # ahora contamos recursivamente los vecinos de cada cuenta
        for value in ego_graph.values():
            count += ego_count(value)
        return count
    
    else:
        raise ValueError(
            f"ego_graph debe ser list o dict. Se recibió {type(ego_graph)}."
        )

In [ ]:
from collections import defaultdict

def ego_stats(ego_graph):
    """
    Calcula:
      - total de nodos (incluyendo el nodo centro),
      - número de nodos por nivel (hop),
      - promedio de vecinos por nivel (hop).

    Se asume:
      - ego_graph es un dict: {cuenta_nivel1: subgrafo_o_lista_nivel2}
      - subgrafo puede ser dict (más niveles) o list (último nivel).
    """
    if not isinstance(ego_graph, dict):
        raise ValueError("La raíz del ego_graph debe ser un dict (nivel 1).")
    
    nodes_per_level = defaultdict(int)
    degree_sum_per_level = defaultdict(int)
    
    # hop 0: el centro (no está en la estructura)
    nodes_per_level[0] = 1
    degree_sum_per_level[0] = len(ego_graph)  # grado del nodo centro
    
    def _walk(current, hop):
        if not isinstance(current, dict):
            raise ValueError(f"Esperaba dict en nivel {hop}, recibí {type(current)}.")
        
        # nodos en este nivel = keys del dict
        nodes_per_level[hop] += len(current)
        
        for sub in current.values():
            if isinstance(sub, dict):
                # vecinos de este nodo (hacia el siguiente nivel)
                deg = len(sub)
                degree_sum_per_level[hop] += deg
                # bajar un nivel
                _walk(sub, hop + 1)
            elif isinstance(sub, list):
                deg = len(sub)
                degree_sum_per_level[hop] += deg
                # nodos del último nivel (hop+1)
                nodes_per_level[hop + 1] += len(sub)
                # no hay recursión, son hojas
            else:
                raise ValueError(
                    f"Valores deben ser dict o list. Se recibió {type(sub)} en nivel {hop}."
                )
    
    # empezamos en nivel 1 (vecinos del centro)
    _walk(ego_graph, hop=1)
    
    total_nodes = sum(nodes_per_level.values())
    
    # promedio de vecinos por nivel, evitando división por cero
    avg_neighbors_per_level = {}
    for hop in sorted(nodes_per_level.keys()):
        n = nodes_per_level[hop]
        if n == 0:
            avg_neighbors_per_level[hop] = 0.0
        else:
            avg_neighbors_per_level[hop] = degree_sum_per_level[hop] / n
    
    return total_nodes, dict(nodes_per_level), avg_neighbors_per_level


In [17]:
cta_intern = (
    df_clientes
        .select(F.col("num_cta").alias("cta"))
        .distinct()
)

In [18]:
edges_all = (
    df_entrada
        .select(
            F.col("cta_ori").alias("cta_ori"),
            F.col("cta_dst").alias("cta_dst"),
            F.lit("entrada").alias("tipo_tx")
        )
        .unionByName(
            df_salida
                .select(
                    F.col("cta_ori").alias("cta_ori"),
                    F.col("cta_dst").alias("cta_dst"),
                    F.lit("salida").alias("tipo_tx")                
                )
        )
        .dropDuplicates(["cta_ori", "cta_dst", "tipo_tx"])
)

In [19]:
edges_with_flags = (
    edges_all
        .join( #Marcar si el origen es interno
            cta_intern.withColumnRenamed("cta", "cta_ori_int"),
            F.col("cta_ori") == F.col("cta_ori_int"),
            "left"
        )
        .join( #Marcar si el destino es interno
            cta_intern.withColumnRenamed("cta", "cta_dst_int"),
            F.col("cta_dst") == F.col("cta_dst_int"),
            "left"
        )
        .withColumn("ori_intern", F.when(F.col("cta_ori_int").isNotNull(), True).otherwise(False))
        .withColumn("dst_intern", F.when(F.col("cta_dst_int").isNotNull(), True).otherwise(False))
        .drop("cta_ori_int", "cta_dst_int")
)

In [20]:
edges_intern = (
    edges_with_flags
        .filter(F.col("ori_intern") == True)
        .filter(F.col("dst_intern") == True)
        .select("cta_ori", "cta_dst", "tipo_tx")
        .dropDuplicates(["cta_ori", "cta_dst", "tipo_tx"])
)

edges_intern.cache()

DataFrame[cta_ori: string, cta_dst: string, tipo_tx: string]

In [21]:
#Grado de salida e ingreso en el subgrafo interno
deg_out = (
    edges_intern
        .groupBy("cta_ori")
        .agg(F.countDistinct("cta_dst").alias("deg_out"))
)

deg_in = (
    edges_intern
        .groupBy("cta_dst")
        .agg(F.countDistinct("cta_ori").alias("deg_in"))
)

#Unir los grados de salida e ingreso
deg = (
    deg_out
        .join(deg_in, deg_out.cta_ori == deg_in.cta_dst, "outer")
        .select(
            F.coalesce(deg_out.cta_ori, deg_in.cta_dst).alias("cta"),
            F.coalesce(deg_out.deg_out, F.lit(0)).alias("deg_out"),
            F.coalesce(deg_in.deg_in, F.lit(0)).alias("deg_in")
        )
        .withColumn("deg", F.col("deg_out") + F.col("deg_in"))
)

deg.cache()

DataFrame[cta: string, deg_out: bigint, deg_in: bigint, deg: bigint]

In [79]:
min_deg = 10

cta_center = (
    deg
        .filter(F.col("deg") >= 20)
        .sample(withReplacement=False, fraction=0.01)
        .limit(100)
)
center_cands = cta_center.toPandas()
center_cands
#8f4855c7f65edb1e51f42b326363f6b6ec3438d5e5aeb9b35ee320d5c7a001ce
#bb4b6c5c4e6392ecbaa08f1aa653f90684c41807816a090664ccfeff70f17091
#4aab5b2f3847918ad84282b32f2588698d0ae5fe28e3d9938b5d404c50adaf2a

,cta,deg_out,deg_in,deg
0,346a8da662fc6fcd80aeecb7ad3de9db1a720e31cb03d4...,41,3,44
1,36ff607589203b0d2b13bb7f4b17d0001208fe212dd393...,8,13,21
2,707e78074ea96c32ca212baadf380e087ae29f6d79ba75...,42,17,59
3,0d98b963051cdd26c271be7538999fc748aca098e89b14...,43,25,68
4,162883f60203550abf62d930ec1f2ed90c145014ccbf37...,42,24,66
...,...,...,...,...
95,8bb5d12138747b6a14efae13a15ab7f657675f402e123e...,23,11,34
96,926ad04974f1e65bfce66d9304b0fdd5b70c73bc02561f...,28,17,45
97,23657dc33aff38e77b4620f371254923a2f027f4ed8f20...,44,23,67
98,2f9b4944a8111a2980f3efdd88e7a48bf76218b31dcc0f...,37,21,58


In [80]:
display(cta_intern.sample(withReplacement=False, fraction=0.001))
display(edges_all.sample(withReplacement=False, fraction=0.001))
display(edges_with_flags.sample(withReplacement=False, fraction=0.001))
display(edges_intern.sample(withReplacement=False, fraction=0.001))  

DataFrame[cta: string]

DataFrame[cta_ori: string, cta_dst: string, tipo_tx: string]

DataFrame[cta_ori: string, cta_dst: string, tipo_tx: string, ori_intern: boolean, dst_intern: boolean]

DataFrame[cta_ori: string, cta_dst: string, tipo_tx: string]

In [66]:
ego_graph = get_ego_graph(edges_intern, '67cbdee2aaad33ed20ccf8c7c1f5ba3a9c4377b4d5e05d055d090565cc4937eb', 'out', 2)

In [67]:
ego_graph

{'faabc94835a19bc7683dc8daa1dad3316cfb6d7e617db33f99d263ce967fa7f6': ['b1c05025e0df7f4099132df71d9b3f518d1c9be2b16b25f8430d15dc65d71652'],
 '43ebc60a95baf0df390fcdcec0ffc9f171ef00b01a0ce71b784ec1e367a2866f': [],
 '0c4e5d52bc3f26ff6b9e3d1be626361ef274ccd847c5c42543836e7b8a37c5be': [],
 'd17ca9f4dc96b6d653bc871c7beed6dff50a757c654a463927b330883423584a': [],
 'fb9237b5139ea0b750bf071e5b00a1e29d2311d05c8687807092013d1572dbd1': [],
 '02cceab4d20b312d9751781fcfbcf7f41620764e801b716e112692a3031650a5': ['096226692acbb5385d27f5cb1a1b9dcdb905d7a2c44e7103cee8fc7b236c8ba5',
  '02fb8d1a612c62d4da5ef0bcab1d8601e76b17d0192582a12ce794a67fc67644',
  'e67e64b8f05b6c9797b374d940fa7a07d42ba9924cd6ff327d18ea627bc4ba2b',
  'ab57ba1bc1db1e41fdc3d4325e88199c8cbd7b73beaaab92371c398a9946f8f8',
  'd51833ef5cc6678b0643794aacacb2c162d6e7385a942837c679ee9b2e8d1902',
  '600036942c4c375cdbdb463634cce238fbb09faae31f5ca56b149f756a20a1e3',
  '670449e8902b73181b9901cfbf2b450906fa1371ab4bf9f13b98586786e18e17',
  '1beacbfd

In [83]:
ego_graph = get_ego_graph(edges_intern, '67cbdee2aaad33ed20ccf8c7c1f5ba3a9c4377b4d5e05d055d090565cc4937eb', 'out', 3)

In [84]:
ego_count(ego_graph)

36

In [87]:
ego_stats(ego_graph)

(37,
 {0: 1, 1: 17, 2: 19, 3: 0},
 {0: 17.0, 1: 1.1176470588235294, 2: 0.0, 3: 0.0})

In [37]:
type(cta_center) == DataFrame

True

In [36]:
from pyspark.sql.dataframe import DataFrame